# Document Intelligence — Exploration Notebook

Use this notebook to:
- Inspect raw Azure Document Intelligence SDK responses
- Prototype field mappings before adding them to `ExtractionResult`
- Compare outputs across different model IDs
- Experiment with engineering drawings, invoices, etc.

In [ ]:
import sys
from pathlib import Path

# Add project root to path if running notebook directly
sys.path.insert(0, str(Path.cwd().parent))

from doc_intel.config import settings
print(f"Endpoint: {settings.azure_document_intelligence_endpoint}")
print(f"Model: {settings.azure_di_model_id}")

In [ ]:
from doc_intel.extractors.azure_doc_intel import AzureDocIntelExtractor

extractor = AzureDocIntelExtractor()

# Change this to a real document path
doc_path = Path("../inputs/sample.pdf")

result = extractor.extract(doc_path)
print(f"Fields: {len(result.fields)}")
print(f"Pages: {len(result.pages)}")
print(f"Tables: {len(result.tables)}")
print(f"Confidence: {result.confidence:.2%}" if result.confidence else "Confidence: N/A")

In [ ]:
# Inspect extracted fields
for field in result.fields:
    conf = f"{field.confidence:.1%}" if field.confidence is not None else "N/A"
    print(f"  {field.name:30s} → {str(field.value):40s} (conf: {conf})")

In [ ]:
# Inspect page content
for page in result.pages:
    print(f"\n── Page {page.page_number} ({page.width}x{page.height} {page.unit}) ──")
    if page.content:
        print(page.content[:500], "..." if len(page.content or "") > 500 else "")

In [ ]:
# Inspect tables
for i, table in enumerate(result.tables):
    print(f"\n── Table {i + 1} ──")
    for row in table:
        print(" | ".join(row))

In [ ]:
# Save result to JSON
from doc_intel.output.json_writer import JsonWriter

writer = JsonWriter(output_dir=Path("../outputs"))
output_path = writer.write(result)
print(f"Saved: {output_path}")